<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# Soluciones — Calidad de código y refactorización

## Del notebook que funciona al pipeline que un equipo puede confiar

**Caso:** un equipo de operaciones genera cada mañana un informe de actividad transaccional por canal y región. El informe de ayer no se publicó: el notebook mostró un mensaje genérico y nadie pudo identificar la causa.

En esta sesión convertiremos una etapa frágil en un pipeline trazable. Seguiremos el ciclo profesional:

```text
provocar → observar → diagnosticar → refactorizar → validar
```

> **Versión docente.** Todas las celdas de práctica están resueltas. Los bloques desplegables se conservan para mostrar el nivel de ayuda disponible en la versión del alumnado.

## Objetivos

Al finalizar podrás:

- distinguir errores estructurales de problemas de calidad de filas;
- decidir cuándo detener un proceso y cuándo continuar dejando trazabilidad;
- configurar un logger que se comporte correctamente al reejecutar un notebook;
- separar validación, limpieza, transformación y resumen;
- comprobar que una refactorización conserva el contrato del pipeline.

In [46]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Final

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

SEED: Final = 42
UMBRAL_REVISION: Final = 5_000
CANALES_CONOCIDOS: Final = {"movil", "web", "oficina"}
COLUMNAS_OBLIGATORIAS: Final = {
    "operacion_id", "cliente_id", "fecha_operacion", "canal",
    "region", "importe", "pais_destino",
}

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
print("Entorno preparado.")

Entorno preparado.


# Recorrido común

## Un extracto pequeño, pero representativo

El dataset es sintético. Representa operaciones de clientes y contiene las variables necesarias para el informe. Más adelante el mismo contrato de datos se reutilizará en las sesiones de rendimiento y escalabilidad.

In [47]:
def generar_operaciones(n: int = 400, seed: int = SEED) -> pd.DataFrame:
    """Genera un extracto sintético de operaciones para el laboratorio."""
    rng = np.random.default_rng(seed)
    fechas = pd.Timestamp("2026-04-01") + pd.to_timedelta(
        rng.integers(0, 30, size=n), unit="D"
    )
    return pd.DataFrame(
        {
            "operacion_id": np.arange(10_000, 10_000 + n),
            "cliente_id": rng.integers(1_000, 1_120, size=n),
            "fecha_operacion": fechas.astype(str),
            "canal": rng.choice(["movil", "web", "oficina"], size=n, p=[0.55, 0.35, 0.10]),
            "region": rng.choice(["norte", "centro", "este", "sur"], size=n),
            "importe": rng.gamma(shape=2.2, scale=850, size=n).round(2),
            "pais_destino": rng.choice(["ES", "PT", "FR", "GB", "US"], size=n, p=[0.72, 0.08, 0.08, 0.07, 0.05]),
        }
    )


operaciones_validas = generar_operaciones()
operaciones_validas.head()

,operacion_id,cliente_id,fecha_operacion,canal,region,importe,pais_destino
0,10000,1119,2026-04-03,movil,centro,871.40,GB
1,10001,1093,2026-04-24,oficina,este,"1,627.20",ES
2,10002,1038,2026-04-20,web,sur,"1,466.52",ES
3,10003,1116,2026-04-14,movil,centro,"2,416.19",ES
4,10004,1059,2026-04-13,oficina,norte,658.40,PT


In [48]:
def introducir_incidencias(operaciones: pd.DataFrame) -> pd.DataFrame:
    """Crea un extracto con incidencias habituales de calidad."""
    salida = operaciones.copy(deep=True)
    salida.loc[3, "importe"] = -125.50
    salida.loc[7, "fecha_operacion"] = "fecha-no-valida"
    salida.loc[11, "cliente_id"] = np.nan
    salida.loc[15, "canal"] = "chat"
    salida.loc[18, "importe"] = np.nan
    return salida


operaciones_problematicas = introducir_incidencias(operaciones_validas)
operaciones_problematicas.loc[[3, 7, 11, 15, 18]]

,operacion_id,cliente_id,fecha_operacion,canal,region,importe,pais_destino
3,10003,"1,116.00",2026-04-14,movil,centro,-125.50,ES
7,10007,"1,017.00",fecha-no-valida,web,este,390.25,ES
11,10011,NaN,2026-04-30,web,norte,"2,397.02",US
15,10015,"1,081.00",2026-04-24,chat,norte,"2,032.55",ES
18,10018,"1,035.00",2026-04-26,movil,norte,NaN,PT


## El notebook heredado

El siguiente código mezcla transformación, control de errores, reglas de negocio y salida. Ejecútalo sobre el extracto con incidencias.

Antes de hacerlo, responde: **¿qué información necesitaría un compañero para poder resolver la incidencia?**

In [49]:
def informe_heredado(operaciones: pd.DataFrame) -> pd.DataFrame | None:
    print("Inicio del informe")
    try:
        datos = operaciones.copy()
        datos["fecha_operacion"] = pd.to_datetime(datos["fecha_operacion"], errors="raise")
        datos["requiere_revision"] = datos.apply(
            lambda fila: fila["importe"] >= 5_000 or fila["pais_destino"] not in {"ES", "PT", "FR"},
            axis=1,
        )
        resumen = (
            datos.groupby(["region", "canal"], as_index=False)
            .agg(operaciones=("operacion_id", "size"), importe_total=("importe", "sum"))
        )
        print("Informe generado")
        return resumen
    except Exception:
        print("El informe no se ha podido generar")
        return None


informe_heredado(operaciones_problematicas)

Inicio del informe
El informe no se ha podido generar


### Diagnóstico

El mensaje final no responde a preguntas operativas básicas: qué fichero o etapa falló, cuántas filas eran inválidas, cuál fue la excepción original o qué resultado parcial se había producido. La refactorización debe hacer visibles esas decisiones.

## Contrato de datos y política de incidencias

No todas las incidencias tienen el mismo tratamiento. En este laboratorio acordamos:

| Incidencia | Decisión | Evidencia |
|---|---|---|
| Falta una columna obligatoria | Detener | Excepción de dominio + log `ERROR` |
| Fecha, cliente o importe inválido en una fila | Excluir fila | Conteo en log `WARNING` |
| Canal no reconocido | Conservar como `otro` | Conteo en log `WARNING` |
| Inicio, fin y métricas de cada etapa | Continuar | Log `INFO` |

In [50]:
def crear_logger(nombre: str) -> logging.Logger:
    """Crea un logger repetible al reejecutar un notebook."""
    logger = logging.getLogger(nombre)
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.propagate = False

    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))
    logger.addHandler(handler)
    return logger


logger = crear_logger("pipeline_operaciones")
logger.info("Logger listo para registrar la ejecución")
logger.debug("Este mensaje no aparece todavía en consola")

INFO | pipeline_operaciones | Logger listo para registrar la ejecución


## Separar responsabilidades

La refactorización no consiste solo en dividir código en funciones. Cada función debe tener una responsabilidad y un contrato claros. Empezamos por el error que impide ejecutar cualquier etapa: un input que no tiene la estructura esperada.

In [51]:
class ContratoDatosError(ValueError):
    """El extracto no cumple el contrato mínimo del pipeline."""


def validar_estructura(operaciones: pd.DataFrame) -> None:
    """Valida columnas obligatorias e identificadores únicos."""
    faltantes = COLUMNAS_OBLIGATORIAS.difference(operaciones.columns)
    if faltantes:
        mensaje = f"Faltan columnas obligatorias: {sorted(faltantes)}"
        logger.error(mensaje)
        raise ContratoDatosError(mensaje)

    duplicados = int(operaciones["operacion_id"].duplicated().sum())
    if duplicados:
        mensaje = f"Se han detectado {duplicados} identificadores de operación duplicados"
        logger.error(mensaje)
        raise ContratoDatosError(mensaje)

    logger.info("Contrato estructural validado | filas=%d", len(operaciones))


validar_estructura(operaciones_problematicas)

INFO | pipeline_operaciones | Contrato estructural validado | filas=400


In [52]:
def diagnosticar_calidad(operaciones: pd.DataFrame) -> pd.DataFrame:
    """Resume las incidencias de calidad antes de aplicar correcciones."""
    importe = pd.to_numeric(operaciones["importe"], errors="coerce")
    fecha = pd.to_datetime(operaciones["fecha_operacion"], errors="coerce")
    diagnostico = {
        "cliente_id ausente": int(operaciones["cliente_id"].isna().sum()),
        "fecha inválida": int(fecha.isna().sum()),
        "importe ausente": int(importe.isna().sum()),
        "importe negativo": int((importe < 0).sum()),
        "canal no reconocido": int((~operaciones["canal"].isin(CANALES_CONOCIDOS)).sum()),
    }
    return pd.DataFrame(diagnostico.items(), columns=["incidencia", "filas"])


diagnosticar_calidad(operaciones_problematicas)

,incidencia,filas
0,cliente_id ausente,1
1,fecha inválida,1
2,importe ausente,1
3,importe negativo,1
4,canal no reconocido,1


### Interpretación

Diagnosticar no es todavía limpiar. Separar ambas acciones permite revisar la magnitud del problema, acordar la política con negocio y conservar una evidencia de lo que se descartó.

In [53]:
def limpiar_operaciones(operaciones: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia filas inválidas y devuelve un registro resumido de incidencias."""
    datos = operaciones.copy(deep=True)
    datos["fecha_operacion"] = pd.to_datetime(datos["fecha_operacion"], errors="coerce")
    datos["importe"] = pd.to_numeric(datos["importe"], errors="coerce")

    canales_desconocidos = ~datos["canal"].isin(CANALES_CONOCIDOS)
    n_canales = int(canales_desconocidos.sum())
    if n_canales:
        logger.warning("Canales no reconocidos reclasificados como otro | filas=%d", n_canales)
        datos.loc[canales_desconocidos, "canal"] = "otro"

    fila_invalida = (
        datos["cliente_id"].isna()
        | datos["fecha_operacion"].isna()
        | datos["importe"].isna()
        | datos["importe"].lt(0)
    )
    descartadas = int(fila_invalida.sum())
    if descartadas:
        logger.warning("Filas excluidas por calidad | filas=%d", descartadas)

    registro = diagnosticar_calidad(operaciones)
    registro.loc[len(registro)] = ["filas excluidas", descartadas]
    registro.loc[len(registro)] = ["filas conservadas", int((~fila_invalida).sum())]
    return datos.loc[~fila_invalida].copy(), registro


operaciones_limpias, registro_calidad = limpiar_operaciones(operaciones_problematicas)
registro_calidad

WARNING | pipeline_operaciones | Canales no reconocidos reclasificados como otro | filas=1
WARNING | pipeline_operaciones | Filas excluidas por calidad | filas=4


,incidencia,filas
0,cliente_id ausente,1
1,fecha inválida,1
2,importe ausente,1
3,importe negativo,1
4,canal no reconocido,1
5,filas excluidas,4
6,filas conservadas,396


In [54]:
def clasificar_revision(operaciones: pd.DataFrame, umbral: float = UMBRAL_REVISION) -> pd.DataFrame:
    """Marca operaciones que requieren revisión manual."""
    datos = operaciones.copy(deep=True)
    destino_sensible = ~datos["pais_destino"].isin({"ES", "PT", "FR"})
    datos["requiere_revision"] = (datos["importe"] >= umbral) | destino_sensible
    logger.info("Regla de revisión aplicada | operaciones=%d", len(datos))
    return datos


def resumir_operaciones(operaciones: pd.DataFrame) -> pd.DataFrame:
    """Agrega las métricas que necesita el informe diario."""
    resumen = (
        operaciones.groupby(["region", "canal"], as_index=False)
        .agg(
            operaciones=("operacion_id", "size"),
            importe_total=("importe", "sum"),
            porcentaje_revision=("requiere_revision", "mean"),
        )
        .sort_values(["importe_total", "region"], ascending=[False, True])
        .reset_index(drop=True)
    )
    logger.info("Resumen generado | grupos=%d", len(resumen))
    return resumen


In [55]:
def ejecutar_pipeline(operaciones: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Orquesta las etapas del informe diario de operaciones."""
    logger.info("Inicio de pipeline")
    validar_estructura(operaciones)
    operaciones_limpias, registro_calidad = limpiar_operaciones(operaciones)
    operaciones_clasificadas = clasificar_revision(operaciones_limpias)
    resumen = resumir_operaciones(operaciones_clasificadas)
    logger.info("Fin de pipeline | filas_procesadas=%d", len(operaciones_clasificadas))
    return {
        "resumen": resumen,
        "calidad": registro_calidad,
        "operaciones_limpias": operaciones_clasificadas,
    }


entrada_original = operaciones_problematicas.copy(deep=True)
resultado = ejecutar_pipeline(operaciones_problematicas)
assert_frame_equal(operaciones_problematicas, entrada_original)
assert resultado["operaciones_limpias"]["importe"].ge(0).all()
assert resultado["operaciones_limpias"]["cliente_id"].notna().all()
resultado["resumen"].head()

INFO | pipeline_operaciones | Inicio de pipeline
INFO | pipeline_operaciones | Contrato estructural validado | filas=400
WARNING | pipeline_operaciones | Canales no reconocidos reclasificados como otro | filas=1
WARNING | pipeline_operaciones | Filas excluidas por calidad | filas=4
INFO | pipeline_operaciones | Regla de revisión aplicada | operaciones=396
INFO | pipeline_operaciones | Resumen generado | grupos=13
INFO | pipeline_operaciones | Fin de pipeline | filas_procesadas=396


,region,canal,operaciones,importe_total,porcentaje_revision
0,norte,movil,59,"110,225.83",0.22
1,centro,movil,50,"96,926.84",0.06
2,sur,movil,55,"95,380.84",0.24
3,este,movil,52,"94,655.18",0.13
4,sur,web,37,"75,055.31",0.16


### Validación

La refactorización ha conservado la entrada, ha separado las etapas y ha dejado una tabla de calidad. Esto no significa que el pipeline esté optimizado: significa que ya disponemos de una base fiable que podremos medir en la siguiente sesión.

## Herramientas que usarás en la práctica

Antes de empezar, estas son las operaciones nuevas que aparecen en los ejercicios. Todas trabajan sobre una serie completa, sin recorrer fila a fila:

- `np.where(condicion, valor_si_verdadero, valor_si_falso)` crea una salida binaria a partir de una máscara booleana.
- `serie.isin(valores)` construye una máscara que indica qué valores pertenecen al conjunto. `serie.where(condicion, otro_valor)` conserva los valores que cumplen la condición y sustituye el resto.
- `np.select([condicion_1, condicion_2], [valor_1, valor_2], default=...)` generaliza `np.where` cuando hay más de dos resultados. Se evalúa de izquierda a derecha.
- `assert condicion` detiene la ejecución si una comprobación no se cumple; se usa para fijar el comportamiento esperado de una regla.


In [56]:
canales_ejemplo = pd.Series(["movil", "chat", "web"])
es_conocido = canales_ejemplo.isin(CANALES_CONOCIDOS)
canales_normalizados = canales_ejemplo.where(es_conocido, "otro")

importes_ejemplo = pd.Series([3_500, 8_500])
prioridad_binaria = np.where(importes_ejemplo >= 8_000, "alta", "normal")
prioridad_tres_niveles = np.select(
    [importes_ejemplo >= 8_000, importes_ejemplo >= 5_000],
    ["alta", "estándar"],
    default="ninguna",
)

assert prioridad_binaria[1] == "alta"
display(pd.DataFrame({"canal_original": canales_ejemplo, "canal_normalizado": canales_normalizados}))
display(pd.DataFrame({"importe": importes_ejemplo, "prioridad": prioridad_tres_niveles}))

,canal_original,canal_normalizado
0,movil,movil
1,chat,otro
2,web,web


,importe,prioridad
0,3500,ninguna
1,8500,alta


# Ejercicios prácticos: recuperar el informe diario

Los ejercicios siguen el orden natural de una incidencia: interpretar el problema, decidir si el informe se publica, registrar lo ocurrido, reforzar el contrato y proteger las reglas para futuras ejecuciones.

En cada misión encontrarás una **celda vacía**. Intenta resolverla desde cero usando el código visto antes. Si te bloqueas, despliega **Obtener esqueleto de ayuda**, copia el código en la celda vacía y completa únicamente los puntos marcados con `TODO`.

## Ejercicio 1 — Interpretar la incidencia antes de cambiar código

Usa `resultado["calidad"]` y responde en una celda Markdown:

1. ¿Qué incidencias se han corregido sin detener el pipeline?
2. ¿Qué incidencia, si afectara a todas las filas, debería obligar a detenerlo?
3. ¿Por qué el canal `chat` se conserva mientras que un importe negativo se descarta?

No hay una única redacción correcta; debes justificar la decisión con el contrato acordado.

## Ejercicio 2 — Convertir calidad de datos en una decisión operativa

El responsable del informe no necesita una lista técnica de incidencias. Necesita saber si la ejecución puede publicarse. Completa `evaluar_calidad_ejecucion()` para devolver un diccionario con:

- `filas_excluidas`;
- `tasa_exclusion`;
- `estado`: `aceptable` si la tasa de exclusiones no supera el 2 %, y `revisar` en caso contrario.

**Dato de control:** en el extracto actual se excluyen 4 de 400 filas; el estado esperado es `aceptable`.

In [57]:
def evaluar_calidad_ejecucion(registro: pd.DataFrame, total_entrada: int) -> dict[str, float | int | str]:
    """Resume si la calidad observada permite publicar el informe diario."""
    filas_excluidas = int(
        registro.loc[registro["incidencia"] == "filas excluidas", "filas"].iloc[0]
    )
    tasa_exclusion = filas_excluidas / total_entrada
    estado = "aceptable" if tasa_exclusion <= 0.02 else "revisar"
    return {
        "filas_excluidas": filas_excluidas,
        "tasa_exclusion": tasa_exclusion,
        "estado": estado,
    }

<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 2</strong></summary>

    ```python
    def evaluar_calidad_ejecucion(registro: pd.DataFrame, total_entrada: int) -> dict:
        filas_excluidas = int(
            registro.loc[registro["incidencia"] == "filas excluidas", "filas"].iloc[0]
        )
        tasa_exclusion = filas_excluidas / total_entrada
        # TODO: decide el estado con un umbral de exclusión del 2 %.
        estado = ...
        return {"filas_excluidas": filas_excluidas, "tasa_exclusion": tasa_exclusion, "estado": estado}
    ```
    </details>

## Ejercicio 3 — Nueva regla de revisión priorizada

Operaciones quiere una señal muy sencilla para empezar: distinguir solo las operaciones de **alta prioridad** del resto. Define la siguiente política:

- **alta**: importe igual o superior a 8.000 EUR, o destino `US`;
- **normal**: el resto.

La máscara `es_alta` ya está construida. Completa una única línea con `np.where()`. No modifiques el DataFrame de entrada.

In [58]:
def asignar_prioridad_revision(operaciones: pd.DataFrame) -> pd.DataFrame:
    """Devuelve una copia con prioridad_revision: alta o normal."""
    datos = operaciones.copy(deep=True)
    es_alta = (datos["importe"] >= 8_000) | datos["pais_destino"].eq("US")
    datos["prioridad_revision"] = np.where(es_alta, "alta", "normal")
    return datos

<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 3</strong></summary>

```python
def asignar_prioridad_revision(operaciones: pd.DataFrame) -> pd.DataFrame:
    datos = operaciones.copy(deep=True)
    es_alta = (datos["importe"] >= 8_000) | datos["pais_destino"].eq("US")
    # TODO: usa np.where con es_alta para asignar "alta" o "normal".
    datos["prioridad_revision"] = ...
    return datos
```
</details>

## Sanity check — ¿funciona tu solución?

Después de completar los ejercicios 2 y 3, ejecuta la celda siguiente. No es una solución: comprueba que tu implementación cumple el contrato mínimo y genera un resumen para poder inspeccionarlo.

Si aparece el mensaje de que faltan funciones, vuelve a las celdas vacías, completa tu código y ejecútalas antes de repetir esta comprobación.

In [59]:
funciones_necesarias = {"evaluar_calidad_ejecucion", "asignar_prioridad_revision"}

if not funciones_necesarias.issubset(globals()):
    print("Completa y ejecuta primero los ejercicios 2 y 3.")
else:
    entrada_sanity = resultado["operaciones_limpias"].copy(deep=True)
    decision_publicacion = evaluar_calidad_ejecucion(
        resultado["calidad"], total_entrada=len(operaciones_problematicas)
    )
    operaciones_priorizadas = asignar_prioridad_revision(entrada_sanity)

    assert_frame_equal(entrada_sanity, resultado["operaciones_limpias"])
    assert decision_publicacion["estado"] == "aceptable"
    assert set(operaciones_priorizadas["prioridad_revision"]) <= {"alta", "normal"}

    resumen_prioridades = (
        operaciones_priorizadas.groupby(["region", "canal", "prioridad_revision"], as_index=False)
        .agg(operaciones=("operacion_id", "size"), importe_total=("importe", "sum"))
        .sort_values("importe_total", ascending=False)
        .reset_index(drop=True)
    )

    display(pd.DataFrame([decision_publicacion]))
    display(resumen_prioridades.head())
    print("Sanity check superado: la entrada se conserva y el informe es coherente.")

,filas_excluidas,tasa_exclusion,estado
0,4,0.01,aceptable


,region,canal,prioridad_revision,operaciones,importe_total
0,norte,movil,normal,54,"100,972.44"
1,centro,movil,normal,50,"96,926.84"
2,sur,movil,normal,51,"88,037.17"
3,este,movil,normal,48,"84,878.13"
4,centro,web,normal,33,"69,993.84"


Sanity check superado: la entrada se conserva y el informe es coherente.


### Comprobación tras el ejercicio 3

Tu solución debe demostrar que:

- la entrada se conserva sin cambios;
- toda operación recibe una prioridad `alta` o `normal`;
- las operaciones de alta prioridad cumplen la regla acordada;
- la tasa de exclusión y el estado de publicación son correctos;
- el sanity check genera un resumen por región, canal y prioridad;
- el log permite explicar cuántas filas se descartaron y por qué.

Cuando termines, comparte una recomendación de una frase para el equipo de operaciones: ¿publicarías el informe actual o solicitarías una revisión?

## Ejercicio 4 — Elegir el nivel de logging

Completa el diccionario `nivel_por_incidencia`. Usa `INFO` cuando el pipeline informa de una etapa normal, `WARNING` cuando puede continuar pero hay que revisar una incidencia y `ERROR` cuando no puede seguir.

Después, ejecuta el bucle que has creado y explica por qué cada evento ha recibido ese nivel.

In [60]:
incidencias_demo = [
    ("inicio de pipeline", 400),
    ("filas excluidas por calidad", 4),
    ("falta una columna obligatoria", 1),
]

nivel_por_incidencia = {
    "inicio de pipeline": logger.info,
    "filas excluidas por calidad": logger.warning,
    "falta una columna obligatoria": logger.error,
}

for incidencia, filas in incidencias_demo:
    nivel_por_incidencia[incidencia]("%s | filas=%d", incidencia, filas)

INFO | pipeline_operaciones | inicio de pipeline | filas=400
WARNING | pipeline_operaciones | filas excluidas por calidad | filas=4
ERROR | pipeline_operaciones | falta una columna obligatoria | filas=1


<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 4</strong></summary>

```python
incidencias_demo = [("inicio de pipeline", 400), ("filas excluidas por calidad", 4), ("falta una columna obligatoria", 1)]
nivel_por_incidencia = {
    # TODO: asigna el método de logger adecuado a cada incidencia.
}
for incidencia, filas in incidencias_demo:
    # TODO: emite aquí el mensaje usando el nivel elegido.
```
</details>

## Ejercicio 5 — Comprobar que el contrato detiene el proceso

Crea un extracto sin la columna `region` y ejecuta `validar_estructura()` dentro de un bloque `try/except ContratoDatosError`.

El bloque debe imprimir un mensaje comprensible para operaciones y dejar que el logger conserve el detalle. ¿Por qué este caso no se resuelve descartando filas?

In [61]:
extracto_sin_region = operaciones_validas.drop(columns="region")

try:
    validar_estructura(extracto_sin_region)
except ContratoDatosError as error:
    mensaje_operaciones = f"No se puede publicar el informe: {error}"
    print(mensaje_operaciones)

ERROR | pipeline_operaciones | Faltan columnas obligatorias: ['region']


No se puede publicar el informe: Faltan columnas obligatorias: ['region']


<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 5</strong></summary>

```python
extracto_sin_region = operaciones_validas.drop(columns="region")
try:
    # TODO: valida la estructura del extracto.
    ...
except ContratoDatosError as error:
    # TODO: crea e imprime un mensaje comprensible para operaciones.
    ...
```
</details>

## Ejercicio 6 — Extraer una regla reutilizable

La normalización de canales está actualmente dentro de `limpiar_operaciones()`. Extrae esa responsabilidad a una función reusable. La máscara ya está creada: completa una sola expresión con `where()`.

Verifica que `chat` y cualquier otro canal no reconocido se convierten en `otro`, y que la serie original no cambia.

In [62]:
def normalizar_canales(canal: pd.Series) -> pd.Series:
    """Mantiene canales conocidos y agrupa el resto como otro."""
    es_conocido = canal.isin(CANALES_CONOCIDOS)
    return canal.where(es_conocido, "otro")


canales_prueba = pd.Series(["movil", "chat", "oficina", "telefono"])
normalizar_canales(canales_prueba)

0      movil
1       otro
2    oficina
3       otro
dtype: str

<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 6</strong></summary>

```python
def normalizar_canales(canal: pd.Series) -> pd.Series:
    es_conocido = canal.isin(CANALES_CONOCIDOS)
    # TODO: devuelve los canales conocidos y sustituye el resto por "otro".
    return ...

canales_prueba = pd.Series(["movil", "chat", "oficina", "telefono"])
# TODO: llama a la función y comprueba que la serie original no cambia.
```
</details>

## Ejercicio 7 — Añadir una prioridad intermedia

Amplía la regla de prioridad para distinguir tres estados:

- `alta`: importe mayor o igual que 8.000 EUR o destino `US`;
- `estándar`: requiere revisión, pero no es alta;
- `ninguna`: el resto.

Las dos máscaras ya están preparadas. Completa `np.select()` y comprueba que cada operación recibe una única prioridad. Para probarla, usa `resultado["operaciones_limpias"]`, que ya contiene la columna `requiere_revision`.

In [63]:
def asignar_prioridad_avanzada(operaciones: pd.DataFrame) -> pd.DataFrame:
    """Asigna una prioridad alta, estándar o ninguna."""
    datos = operaciones.copy(deep=True)
    es_alta = (datos["importe"] >= 8_000) | datos["pais_destino"].eq("US")
    es_estandar = datos["requiere_revision"] & ~es_alta
    datos["prioridad_revision"] = np.select(
        [es_alta, es_estandar],
        choices=["alta", "estándar"],
        default="ninguna",
    )
    return datos

<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 7</strong></summary>

```python
def asignar_prioridad_avanzada(operaciones: pd.DataFrame) -> pd.DataFrame:
    datos = operaciones.copy(deep=True)
    es_alta = (datos["importe"] >= 8_000) | datos["pais_destino"].eq("US")
    es_estandar = datos["requiere_revision"] & ~es_alta
    # TODO: usa np.select para combinar las dos máscaras y el caso por defecto.
    datos["prioridad_revision"] = ...
    return datos
```
</details>

## Ejercicio 8 — Proteger una regla con una prueba

Un cambio futuro no debe convertir una operación de 8.000 EUR en prioridad normal. Completa los dos `assert` siguientes y explica por qué este tipo de comprobación evita una regresión.

El ejercicio no requiere `pytest` todavía: usa `assert` como anticipo de los tests que se trabajarán en el proyecto final.

In [64]:
casos_limite = pd.DataFrame(
    {
        "importe": [7_999, 8_000],
        "pais_destino": ["ES", "ES"],
    }
)
resultado_limite = asignar_prioridad_revision(casos_limite)
assert resultado_limite.loc[0, "prioridad_revision"] == "normal"
assert resultado_limite.loc[1, "prioridad_revision"] == "alta"
print("Prueba de regresión superada.")

Prueba de regresión superada.


<details>
<summary><strong>Obtener esqueleto de ayuda para el ejercicio 8</strong></summary>

```python
casos_limite = pd.DataFrame({
    "importe": [7_999, 8_000],
    "pais_destino": ["ES", "ES"],
})
resultado_limite = asignar_prioridad_revision(casos_limite)
# TODO: comprueba el caso 7.999 EUR.
assert ...
# TODO: comprueba el caso 8.000 EUR.
assert ...
```
</details>

Los ejercicios 4 a 8 recuperan la progresión del notebook base: niveles de logging, errores específicos, extracción de funciones, reglas de negocio y validación. Pueden usarse como trabajo individual, en parejas o como práctica posterior a la sesión.

# Extensión: cuando falla la fuente de datos

La práctica anterior empieza con un DataFrame ya cargado. En un entorno real, una incidencia habitual ocurre antes: el fichero diario no llega, está vacío o tiene un formato inválido.

El objetivo de esta extensión es doble: dejar evidencia persistente en un log y comunicar un error comprensible al equipo de operaciones, sin perder la causa técnica que necesitaría desarrollo.

In [65]:
class FuenteDatosError(RuntimeError):
    """No se ha podido obtener una fuente necesaria para el pipeline."""


def activar_log_a_fichero(ruta: Path) -> None:
    """Añade un FileHandler una sola vez al logger del pipeline."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if any(isinstance(handler, logging.FileHandler) for handler in logger.handlers):
        return
    handler = logging.FileHandler(ruta, encoding="utf-8")
    handler.setLevel(logging.DEBUG)
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(handler)


def cargar_operaciones_csv(ruta: Path) -> pd.DataFrame:
    """Carga y valida un CSV, traduciendo errores técnicos al dominio del pipeline."""
    try:
        operaciones = pd.read_csv(ruta)
    except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
        logger.exception("No se ha podido cargar la fuente | ruta=%s", ruta)
        raise FuenteDatosError(
            f"No está disponible un extracto válido de operaciones: {ruta.name}"
        ) from error

    validar_estructura(operaciones)
    logger.info("Fuente cargada correctamente | ruta=%s | filas=%d", ruta, len(operaciones))
    return operaciones

In [66]:
ruta_log = Path("artefactos") / "pipeline_operaciones.log"
activar_log_a_fichero(ruta_log)
ruta_ausente = Path("datos_entrada") / "operaciones_2026-04-30.csv"

try:
    cargar_operaciones_csv(ruta_ausente)
except FuenteDatosError as error:
    print(f"Mensaje para operaciones: {error}")
    print(f"Causa técnica conservada: {type(error.__cause__).__name__}")

print(f"Log de auditoría: {ruta_log.resolve()}")

ERROR | pipeline_operaciones | No se ha podido cargar la fuente | ruta=datos_entrada/operaciones_2026-04-30.csv
Traceback (most recent call last):
  File "/var/folders/3z/w0gc4qzj2_v1qf_lvky61n6w0000gn/T/ipykernel_37133/2791092985.py", line 19, in cargar_operaciones_csv
    operaciones = pd.read_csv(ruta)
  File "/Users/andrea/Library/CloudStorage/SynologyDrive-Dory/classes/postgrau/Python Profesional para Analtica Avanzada/.venv/lib/python3.14/site-packages/pandas/io/parsers/readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/Users/andrea/Library/CloudStorage/SynologyDrive-Dory/classes/postgrau/Python Profesional para Analtica Avanzada/.venv/lib/python3.14/site-packages/pandas/io/parsers/readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/Users/andrea/Library/CloudStorage/SynologyDrive-Dory/classes/postgrau/Python Profesional para Analtica Avanzada/.venv/lib/python3.14/site-packages/pandas/io/parsers/reade

Mensaje para operaciones: No está disponible un extracto válido de operaciones: operaciones_2026-04-30.csv
Causa técnica conservada: FileNotFoundError
Log de auditoría: /Users/andrea/Library/CloudStorage/SynologyDrive-Dory/classes/postgrau/Python Profesional para Analtica Avanzada/caixabank_formacion_andrea/materiales/calidad_refactorizacion/artefactos/pipeline_operaciones.log


# Retos opcionales

1. **Nuevo canal confirmado.** El área de negocio confirma que `chat` es un canal válido y quiere analizarlo por separado. Modifica la política para conservar `chat`, pero seguir reclasificando cualquier otro canal desconocido como `otro`. Verifica el número de operaciones por canal y registra una advertencia solo para canales realmente desconocidos.

2. **Monitorización de calidad.** A partir de `resultado["calidad"]`, crea una tabla con porcentajes sobre el total de entrada y añade una columna `alerta`. Debe marcar `True` cuando la tasa de exclusión supere el 2 %. Formula una recomendación sobre qué métrica debería revisarse cada mañana.

In [67]:
CANALES_CONFIRMADOS = CANALES_CONOCIDOS | {"chat"}


def normalizar_canales_confirmados(canal: pd.Series) -> pd.Series:
    """Conserva chat como canal válido y normaliza otros valores nuevos."""
    desconocidos = ~canal.isin(CANALES_CONFIRMADOS)
    if desconocidos.any():
        logger.warning("Canales todavía desconocidos | filas=%d", int(desconocidos.sum()))
    return canal.where(~desconocidos, "otro")


canales_reto = pd.Series(["movil", "chat", "telefono", "chat"])
canales_normalizados = normalizar_canales_confirmados(canales_reto)
display(canales_normalizados.value_counts().rename_axis("canal").reset_index(name="operaciones"))

WARNING | pipeline_operaciones | Canales todavía desconocidos | filas=1


,canal,operaciones
0,chat,2
1,movil,1
2,otro,1


In [68]:
monitor_calidad = resultado["calidad"].copy()
total_entrada = len(operaciones_problematicas)
monitor_calidad["porcentaje_entrada"] = monitor_calidad["filas"] / total_entrada
monitor_calidad["alerta"] = (
    monitor_calidad["incidencia"].eq("filas excluidas")
    & monitor_calidad["porcentaje_entrada"].gt(0.02)
)

display(monitor_calidad)
print("Recomendación: revisar cada mañana la tasa de filas excluidas y su evolución temporal.")

,incidencia,filas,porcentaje_entrada,alerta
0,cliente_id ausente,1,0.00,False
1,fecha inválida,1,0.00,False
2,importe ausente,1,0.00,False
3,importe negativo,1,0.00,False
4,canal no reconocido,1,0.00,False
5,filas excluidas,4,0.01,False
6,filas conservadas,396,0.99,False


Recomendación: revisar cada mañana la tasa de filas excluidas y su evolución temporal.


## Preguntas abiertas

1. ¿Qué diferencia práctica hay entre registrar una advertencia y lanzar una excepción?
2. ¿Por qué una columna obligatoria ausente debe detener el proceso, mientras que una fila inválida puede excluirse?
3. ¿Qué riesgo evita devolver copias en `limpiar_operaciones()` y `clasificar_revision()`?
4. ¿Qué información añadirías a los logs para poder investigar una incidencia ocurrida durante la noche?
5. ¿Cuándo podría ser peligroso normalizar automáticamente un valor desconocido como `otro`?
6. ¿En qué parte del proyecto debería vivir la política de revisión: dentro de un notebook, en una función, en un fichero de configuración o en una tabla de negocio? Justifica tu elección.
7. ¿Cómo comprobarías mediante tests que una futura refactorización no cambia las reglas de exclusión?
8. ¿Qué señal de los logs te haría decidir que el pipeline necesita ser revisado antes de optimizar su rendimiento?

# Cierre

Un pipeline profesional no es el que nunca recibe datos incorrectos. Es el que hace explícito su contrato, registra sus decisiones, protege el resultado y permite diagnosticar una incidencia.

En la siguiente sesión usaremos este mismo pipeline como baseline: mediremos su tiempo, su variabilidad y su cuello de botella antes de decidir qué optimizar.